In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    GRU,
    Dense
)

from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report

from transformers import pipeline

In [2]:
df = pd.read_csv("amazon_cleaned.csv")

print(df.shape)

display(df.head())

(1465, 18)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link,review_clean,review_tokens
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...,look durable charging fine toono complains cha...,"['look', 'durable', 'charging', 'fine', 'toono..."
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...,ordered cable connect phone android auto car c...,"['ordered', 'cable', 'connect', 'phone', 'andr..."
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...,quite durable sturdy good nice product working...,"['quite', 'durable', 'sturdy', 'good', 'nice',..."
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...,good product long wire charge good nice bought...,"['good', 'product', 'long', 'wire', 'charge', ..."
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...,bought instead original apple work fast apple ...,"['bought', 'instead', 'original', 'apple', 'wo..."


In [3]:
# Positive = rating >= 3
# Negative = rating < 3

df["sentiment_label"] = df["rating"].apply(
    lambda x: 1 if x >= 3 else 0
)

print(
    df["sentiment_label"].value_counts()
)

sentiment_label
1    1459
0       6
Name: count, dtype: int64


In [4]:
MAX_WORDS = 5000
MAX_LEN = 100

tokenizer = Tokenizer(
    num_words=MAX_WORDS
)

tokenizer.fit_on_texts(
    df["review_clean"]
)

X = tokenizer.texts_to_sequences(
    df["review_clean"]
)

X = pad_sequences(
    X,
    maxlen=MAX_LEN
)

y = df["sentiment_label"]

print(X.shape)

(1465, 100)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [6]:

lstm_model = Sequential()

lstm_model.add(
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=64,
        input_length=MAX_LEN
    )
)

lstm_model.add(
    LSTM(64)
)

lstm_model.add(
    Dense(1, activation="sigmoid")
)

lstm_model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(),
    metrics=["accuracy"]
)

lstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = lstm_model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/2
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 80ms/step - accuracy: 0.9808 - loss: 0.2722 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 2/2
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.9947 - loss: 0.0352 - val_accuracy: 1.0000 - val_loss: 0.0033


In [8]:
loss, accuracy = lstm_model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", accuracy)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.9966 - loss: 0.0226
Test Accuracy: 0.9965870380401611


## LSTM Model Insights

The LSTM model achieved very high test accuracy when predicting customer sentiment from review text. This demonstrates the effectiveness of sequential deep learning models for NLP classification tasks.

However, the dataset was highly imbalanced, containing significantly more positive reviews than negative reviews. As a result, the high accuracy score may partially reflect the dominance of positive sentiment examples within the dataset.

In [9]:
lstm_model.save("lstm_model.h5")

print("LSTM model saved.")

LSTM model saved.


In [10]:
gru_model = Sequential()

gru_model.add(
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=64,
        input_length=MAX_LEN
    )
)

gru_model.add(
    GRU(64)
)

gru_model.add(
    Dense(1, activation="sigmoid")
)

gru_model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(),
    metrics=["accuracy"]
)

gru_model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=32,
    validation_split=0.2
)

loss, accuracy = gru_model.evaluate(
    X_test,
    y_test
)

print("GRU Accuracy:", accuracy)

Epoch 1/2
30/30 ━━━━━━━━━━━━━━━━━━━━ 10s 137ms/step - accuracy: 0.9840 - loss: 0.3677 - val_accuracy: 1.0000 - val_loss: 1.1794e-04
Epoch 2/2
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.9947 - loss: 0.0512 - val_accuracy: 1.0000 - val_loss: 2.6641e-04
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9966 - loss: 0.0281
GRU Accuracy: 0.9965870380401611


In [11]:
gru_model.save("gru_model.h5")

print("GRU model saved.")

GRU model saved.


In [13]:
# =====================================
# Transformer Text Generation Summary
# =====================================

from transformers import pipeline

# Using text-generation as fallback
generator = pipeline(
    task="text-generation",
    model="gpt2"
)

sample_reviews = " ".join(
    df["review_content"].head(3).tolist()
)

prompt = f"Summarize these customer reviews:\n{sample_reviews}\nSummary:"

result = generator(
    prompt,
    max_new_tokens=50,
    do_sample=False
)

print(result[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarize these customer reviews:
Looks durable Charging is fine tooNo complains,Charging is really fast, good product.,Till now satisfied with the quality.,This is a good product . The charging speed is slower than the original iPhone cable,Good quality, would recommend,https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/81---F1ZgHL._SY88.jpg,Product had worked well till date and was having no issue.Cable is also sturdy enough...Have asked for replacement and company is doing the same...,Value for money I ordered this cable to connect my phone to Android Auto of car. The cable is really strong and the connection ports are really well made. I already has a Micro USB cable from Ambrane and it's still in good shape. I connected my phone to the car using the cable and it got connected well and no issues. I also connected it to the charging port and yes it has Fast Charging support.,It quality is good at this price and the main thing is that i didn't ever thought that this cable wi

In [ ]:
## Transformer Summarization Insights

A pretrained transformer language model was successfully used to generate summaries from multiple customer reviews. The generated summaries captured recurring themes related to product durability, charging quality, value for money, and overall customer satisfaction.

This demonstrates how transformer-based NLP systems can help businesses automatically condense large volumes of customer feedback into concise, actionable insights for both consumers and decision-makers.